In [0]:
from pyspark.sql.functions import when, sum, col, lit

In [0]:
accounts_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(
    "abfss://raw-data@crmsalesuksouthadls2.dfs.core.windows.net/accounts.csv/"
)
data_dictionary_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(
    "abfss://raw-data@crmsalesuksouthadls2.dfs.core.windows.net/data_dictionary.csv/"
)

products_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(
    "abfss://raw-data@crmsalesuksouthadls2.dfs.core.windows.net/products.csv/"
)

sales_pipeline_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(
    "abfss://raw-data@crmsalesuksouthadls2.dfs.core.windows.net/sales_pipeline.csv/"
)

sales_teams_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(
    "abfss://raw-data@crmsalesuksouthadls2.dfs.core.windows.net/sales_teams.csv/"
)

In [0]:
print(accounts_df.columns)
print(data_dictionary_df.columns)
print(products_df.columns)
print(sales_pipeline_df.columns)
print(sales_teams_df.columns)

In [0]:
accounts_df= accounts_df.withColumnRenamed('subsidiary_of' , 'parent_company')
data_dictionary_df = data_dictionary_df.withColumnRenamed('Table', 'table').withColumnRenamed('Field', 'field').withColumnRenamed('Description', 'description')



In [0]:
null_counts_accounts_df = accounts_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in accounts_df.columns])
null_counts_data_dictionary_df  = data_dictionary_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in data_dictionary_df.columns])
null_counts_products_df   = products_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in products_df.columns])
null_counts_sales_pipeline_df   = sales_pipeline_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in sales_pipeline_df.columns])
null_counts_sales_teams_df = sales_teams_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in sales_teams_df.columns])

In [0]:
null_counts_accounts_df.show()
null_counts_sales_pipeline_df.show()


In [0]:
accounts_df.display()

In [0]:
accounts_df = accounts_df.fillna({
    "parent_company" : "independent"
})

sales_pipeline_df = sales_pipeline_df.fillna({
    "account" : "unknown"
})



In [0]:

sales_pipeline_df.show()

In [0]:
# Set the access key for the storage account
storage_account_key = dbutils.secrets.get(
    scope="crmSaleScope",
    key="secretkv"
)
spark.conf.set(
    "fs.azure.account.key.crmsalesuksouthadls2.dfs.core.windows.net",
    storage_account_key
)

accounts_df.write.format("csv").mode('overwrite').option('header', 'true').save(
    "abfss://transformed-data@crmsalesuksouthadls2.dfs.core.windows.net/accounts/"
)

data_dictionary_df.write.format("csv").mode('overwrite').option('header', 'true').save(
    "abfss://transformed-data@crmsalesuksouthadls2.dfs.core.windows.net/data_dictionary/"
)

products_df.write.format("csv").mode('overwrite').option('header', 'true').save(
    "abfss://transformed-data@crmsalesuksouthadls2.dfs.core.windows.net/products/"
)

sales_pipeline_df.write.format("csv").mode('overwrite').option('header', 'true').save(
    "abfss://transformed-data@crmsalesuksouthadls2.dfs.core.windows.net/sales_pipeline/"
)

sales_teams_df.write.format("csv").mode('overwrite').option('header', 'true').save(
    "abfss://transformed-data@crmsalesuksouthadls2.dfs.core.windows.net/sales_teams/"
)